# Course 1: Geometric Deep Learning and Graphs
## Session 4: When depth hurts

**Goal:** plot how similar node embeddings become as GCN layers are added, and see a bottleneck edge by its curvature.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on small built-in or synthetic data. Cells labelled NOT RUN need a library, a dataset download or a GPU; run them on Kaggle or Colab.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

def sbm(sizes, p_in, p_out, rng):
    """Stochastic block model: dense inside a block, sparse between blocks."""
    labels = np.repeat(np.arange(len(sizes)), sizes)
    n = len(labels)
    P = np.where(labels[:, None] == labels[None, :], p_in, p_out)
    A = np.triu(rng.random((n, n)) < P, 1)
    A = (A | A.T).astype(np.float32)
    return A, labels

def norm_adj(A):
    """GCN normalization: D^-1/2 (A + I) D^-1/2."""
    A = A + np.eye(len(A), dtype=A.dtype)
    d = A.sum(1)
    return A / np.sqrt(np.outer(d, d))

class GCN(nn.Module):
    def __init__(self, d_in, d_hid, d_out):
        super().__init__()
        self.l1, self.l2 = nn.Linear(d_in, d_hid), nn.Linear(d_hid, d_out)
    def forward(self, X, A_hat):
        H = F.relu(A_hat @ self.l1(X))
        return A_hat @ self.l2(H)

### 1. Oversmoothing: stack layers and measure similarity
No training. We push features through more and more GCN layers and measure how alike the nodes have become.

In [ ]:
import matplotlib.pyplot as plt
rng = np.random.default_rng(0); torch.manual_seed(0)
A, y = sbm([60, 60], 0.15, 0.01, rng)
A_hat = torch.tensor(norm_adj(A)); L = torch.eye(len(A)) - A_hat
H = torch.randn(len(A), 32); H0 = H.clone()

def mean_cosine(H):
    Hn = F.normalize(H, dim=1); S = Hn @ Hn.T
    return ((S.sum() - len(H)) / (len(H) * (len(H) - 1))).item()

def dirichlet(H):                      # small value = neighbours have nearly equal features
    Hn = H / H.norm()
    return torch.trace(Hn.T @ L @ Hn).item()

sims, energies = [mean_cosine(H)], [dirichlet(H)]
for layer in range(32):
    W = torch.randn(32, 32) / 32 ** 0.5
    H = torch.relu(A_hat @ H @ W)
    sims.append(mean_cosine(H)); energies.append(dirichlet(H))

for k in [0, 1, 2, 4, 8, 16, 32]:
    print(f"layers {k:2d}  mean cosine similarity {sims[k]:.3f}  Dirichlet energy {energies[k]:.4f}")
assert sims[32] > 0.95 and sims[32] > sims[1]
plt.plot(sims); plt.xlabel("GCN layers"); plt.ylabel("mean pairwise cosine similarity"); plt.savefig("oversmoothing.png")

### 2. A residual connection slows it down

In [ ]:
H = H0.clone(); sims_res = [mean_cosine(H)]
torch.manual_seed(1)
for layer in range(32):
    W = torch.randn(32, 32) / 32 ** 0.5
    H = 0.8 * H0 + 0.2 * torch.relu(A_hat @ H @ W)        # keep some of the input (as in APPNP and GCNII)
    sims_res.append(mean_cosine(H))
print("after 32 layers: plain", round(sims[32], 3), "| with initial residual", round(sims_res[32], 3))
assert sims_res[32] < sims[32]

### 3. Oversquashing: find the bottleneck
Two communities joined by one bridge. Messages between the halves must all squeeze through that edge. A simple curvature (augmented Forman) is very negative on such edges:

`curvature(i, j) = 4 - deg(i) - deg(j) + 3 * triangles(i, j)`

In [ ]:
A2, _ = sbm([15, 15], 0.6, 0.0, np.random.default_rng(1))
A2[3, 20] = A2[20, 3] = 1                                    # the bridge
deg = A2.sum(1); tri = A2 @ A2
edges = [(i, j) for i in range(30) for j in range(i + 1, 30) if A2[i, j]]
curv = {(i, j): 4 - deg[i] - deg[j] + 3 * tri[i, j] for i, j in edges}
worst = min(curv, key=curv.get)
print("most negative edge:", worst, "curvature", curv[worst], "| median curvature", np.median(list(curv.values())))
assert worst == (3, 20)

### 4. Rewiring: add edges around the bottleneck and watch the spectral gap grow
The spectral gap (second-smallest eigenvalue of the normalized Laplacian) measures how well information flows across the graph.

In [ ]:
def spectral_gap(A):
    d = A.sum(1); Lsym = np.eye(len(A)) - A / np.sqrt(np.outer(d, d))
    return np.sort(np.linalg.eigvalsh(Lsym))[1]

gap_before = spectral_gap(A2)
A3 = A2.copy()
rng = np.random.default_rng(2)
for _ in range(5):                                           # five extra edges between the two halves
    i, j = rng.integers(0, 15), rng.integers(15, 30)
    A3[i, j] = A3[j, i] = 1
print(f"spectral gap before {gap_before:.4f}  after rewiring {spectral_gap(A3):.4f}")
assert spectral_gap(A3) > gap_before

### TODO
1. Train GCNs with 2, 4, 8, 16 layers on the session 2 graph and plot test accuracy against depth.
2. Repeat section 1 with `p_out = 0.1`. Does smoothing happen faster or slower? Why?
3. Implement one step of curvature-based rewiring: add an edge that closes a triangle around the most negative edge, then recompute the curvatures.